# 视觉信息支持：原提示与中性问法的独立验证

现在直接运行本 notebook。

[实验设置与运行说明](../../../docs/experiment_notes/90_followup_visual_and_relation_use/872970cb876a0334.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready.')


In [ ]:
%%cell_progress 1/13 Mount Drive and locate project
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/13 Choose model and known source ZIPs
MODEL_KEY = 'gemma12b'  # Then qwen27b, in a FRESH GPU runtime.
RUN_ID = 'main'
SOURCE_DEVELOPMENT_RUN_ID = 'main'
SOURCE_DIAGNOSTIC_RUN_ID = 'main'
SOURCE_ZIPS = {
    'development': {'gemma12b': None, 'qwen27b': None},
    'diagnostic': {'gemma12b': None, 'qwen27b': None},
}
EXTRA_EXCLUSION_FILES = []  # Exact additional support.json / scenes.jsonl paths, if needed.
# 96 families, seed, two wordings, conditions and comparisons are fixed below.
# None uses the preceding notebook's original Drive output location.


In [ ]:
%%cell_progress 3/13 Install pinned inference dependencies
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'Pillow==11.3.0',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 4/13 Load embedded validation design
import types, sys
_parent=types.ModuleType("visual_information_support_core")
sys.modules[_parent.__name__]=_parent
exec(compile('"""CPU design, rendering, prompts and family-paired analysis for information support."""\nfrom __future__ import annotations\n\nimport hashlib\nimport itertools\nimport json\nfrom pathlib import Path\nfrom collections import Counter\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFont\n\nSCHEMA = \'visual-information-support-v1\'\nCONDITIONS = (\'image\', \'image_weight\', \'image_distance\', \'image_full\', \'text_full\', \'legacy_image\')\nSTRATA = (\'congruent\', \'balanced\', \'equal_weight_distance\', \'weight_conflict\', \'binding_swap\')\nLABELS = (\'A\', \'B\', \'C\')\nSEMANTICS = (\'left\', \'balanced\', \'right\')\nSEMANTIC_TEXT = {\'left\': \'left side\', \'balanced\': \'equal\', \'right\': \'right side\'}\nRULE_TEXT = (\'Each square is one unit of weight. Distance is measured in tick \'\n             \'intervals from the center pivot. Torque equals weight times distance. \'\n             \'Total torque on a side is the sum over stacks on that side.\')\nREPLY_TEXT = (\'Do not explain your reasoning. Reply with exactly one option letter \'\n              \'and nothing else. Valid replies are: A, B, C.\')\nMODEL_SPECS = {\n    \'gemma12b\': dict(model_id=\'google/gemma-4-12B-it\',\n                    revision=\'707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7\'),\n    \'qwen27b\': dict(model_id=\'Qwen/Qwen3.5-27B\',\n                   revision=\'373612cb4f9288fcf550b526d86d85febedd9801\'),\n}\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,\n                                    allow_nan=False).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef save_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef freeze(path, obj):\n    if Path(path).exists():\n        assert read_json(path) == obj, f\'Frozen configuration changed: {path}. Use a new RUN_ID.\'\n    else:\n        save_json(path, obj)\n    return digest(obj)\n\n\ndef signature(stacks):\n    direct = tuple(sorted((s[\'side\'], int(s[\'distance\']), int(s[\'units\'])) for s in stacks))\n    mirror = tuple(sorted((\'right\' if s == \'left\' else \'left\', d, w) for s, d, w in direct))\n    return min(direct, mirror)\n\n\ndef geometries_in(obj):\n    found = set()\n    if isinstance(obj, dict):\n        if isinstance(obj.get(\'stacks\'), list) and obj[\'stacks\']:\n            stacks = obj[\'stacks\']\n            if all(isinstance(s, dict) and {\'side\', \'distance\', \'units\'} <= s.keys() for s in stacks):\n                found.add(signature(stacks))\n        for key, value in obj.items():\n            if key == \'signatures\' and isinstance(value, list):\n                for sig in value:\n                    found.add(signature([dict(side=s, distance=d, units=w) for s, d, w in sig]))\n            elif key != \'stacks\':\n                found.update(geometries_in(value))\n    elif isinstance(obj, list):\n        for item in obj:\n            found.update(geometries_in(item))\n    return found\n\n\ndef load_history(project, run_root, source_run=\'main\', extra_files=(), progress=print):\n    """Read only explicit old supports/snapshots and shallow dataset manifests; no recursive Drive walk."""\n    project, run_root = Path(project), Path(run_root)\n    lock = run_root / \'historical_exclusion.json\'\n    if lock.exists():\n        old = read_json(lock)\n        assert old[\'source_run\'] == source_run and old[\'extra_files\'] == [str(p) for p in extra_files]\n        for name, sha in old[\'files\'].items():\n            assert file_hash(name) == sha, \'Historical input changed: \' + name\n        progress(f"Reusing frozen exclusions: {len(old[\'signatures\'])} geometries; no directory scan.")\n        return geometries_in(old)\n    paths = set()\n    data = project / \'data\'\n    if data.exists():\n        paths.update(data.glob(\'*/scenes.jsonl\'))\n        if (data / \'scenes.jsonl\').is_file():\n            paths.add(data / \'scenes.jsonl\')\n    results = project / \'results\'\n    if results.exists():\n        # Only top-level experiment directories and this known run, never activation shards.\n        for experiment in results.iterdir():\n            if experiment.is_dir() and experiment.resolve() != run_root.parent.resolve():\n                for folder in (experiment, experiment / source_run):\n                    for name in (\'historical_geometry_snapshot.json\', \'support.json\', \'scenes.jsonl\'):\n                        if (folder / name).is_file():\n                            paths.add(folder / name)\n    for name in extra_files:\n        p = Path(name)\n        assert p.is_file(), f\'EXTRA_HISTORY_FILES must name files: {p}\'\n        paths.add(p)\n    assert paths, \'No historical manifests found. Restore old support/snapshot files, or set EXTRA_HISTORY_FILES.\'\n    signatures, hashes = set(), {}\n    for i, path in enumerate(sorted(paths)):\n        progress(f\'Read historical manifest {i+1}/{len(paths)}: {path.name}\')\n        h = hashlib.sha256()\n        with path.open(\'rb\') as stream:\n            if path.suffix == \'.jsonl\':\n                for line in stream:\n                    h.update(line)\n                    if line.strip():\n                        signatures.update(geometries_in(json.loads(line.decode(\'utf-8-sig\'))))\n            else:\n                raw = stream.read()\n                h.update(raw)\n                signatures.update(geometries_in(json.loads(raw.decode(\'utf-8-sig\'))))\n        hashes[str(path)] = h.hexdigest()\n    assert signatures, \'No usable historical geometry was found in the selected manifests.\'\n    save_json(lock, dict(source_run=source_run, extra_files=[str(p) for p in extra_files],\n                         files=hashes, signatures=[[list(t) for t in sig] for sig in sorted(signatures)],\n                         scope=\'Listed manifests/snapshots only; unseen ZIPs or unlisted runs are not certified disjoint.\'))\n    return signatures\n\n\ndef scene_features(stacks):\n    w = {s: sum(x[\'units\'] for x in stacks if x[\'side\'] == s) for s in (\'left\', \'right\')}\n    torque = {s: sum(x[\'units\'] * x[\'distance\'] for x in stacks if x[\'side\'] == s)\n              for s in (\'left\', \'right\')}\n    delta = torque[\'left\'] - torque[\'right\']\n    weight = w[\'left\'] - w[\'right\']\n    outcome = lambda x: \'left\' if x > 0 else \'right\' if x < 0 else \'balanced\'\n    return dict(left_torque=torque[\'left\'], right_torque=torque[\'right\'], net_torque=delta,\n                weight_score=weight, outcome=outcome(delta), weight_outcome=outcome(weight))\n\n\ndef allocation(n):\n    assert isinstance(n, int) and n >= 32 and n % 8 == 0\n    return dict(congruent=n//8, balanced=n//8, equal_weight_distance=n//4,\n                weight_conflict=n//4, binding_swap=n//4)\n\n\ndef _candidate(rng, stratum):\n    dl, dr = [sorted(rng.choice(np.arange(1, 7), 2, replace=False).tolist()) for _ in range(2)]\n    wl, wr = rng.integers(1, 7, size=(2, 2)).tolist()\n    if stratum == \'equal_weight_distance\':\n        wr = wl.copy()\n        if not (all(a > b for a, b in zip(dl, dr)) or all(a < b for a, b in zip(dl, dr))):\n            return None\n    stacks = [dict(side=side, distance=d, units=w) for side, distances, weights in\n              [(\'left\', dl, wl), (\'right\', dr, wr)] for d, w in zip(distances, weights)]\n    f = scene_features(stacks)\n    if stratum == \'congruent\':\n        distance_delta = sum(dl) - sum(dr)\n        if not (f[\'weight_score\'] * f[\'net_torque\'] > 0 and distance_delta * f[\'net_torque\'] > 0):\n            return None\n    elif stratum == \'balanced\' and f[\'net_torque\'] != 0:\n        return None\n    elif stratum == \'weight_conflict\' and f[\'weight_score\'] * f[\'net_torque\'] >= 0:\n        return None\n    roles = {\'base\': stacks}\n    if stratum == \'binding_swap\':\n        swapped = [dict(s) for s in stacks]\n        swapped[0][\'units\'], swapped[1][\'units\'] = swapped[1][\'units\'], swapped[0][\'units\']\n        if f[\'net_torque\'] * scene_features(swapped)[\'net_torque\'] >= 0:\n            return None\n        roles[\'swapped\'] = swapped\n    return roles\n\n\ndef generate_families(n, seed, excluded, phase):\n    rng = np.random.default_rng(seed)\n    used = set(excluded)\n    families = []\n    for stratum, count in allocation(n).items():\n        accepted = 0\n        for attempt in range(300000):\n            roles = _candidate(rng, stratum)\n            if roles is None:\n                continue\n            sigs = {signature(s) for s in roles.values()}\n            if len(sigs) != len(roles) or sigs & used:\n                continue\n            fid = \'vis_\' + digest([SCHEMA, phase, stratum, sorted(sigs)])[:16]\n            slot_ids = np.random.default_rng(int(digest([fid, \'ids\'])[:16], 16)).permutation(4).tolist()\n            records = []\n            for role, stacks in roles.items():\n                for mirror in (False, True):\n                    actual = [dict(s, stack_id=f\'S{slot_ids[i]+1}\',\n                                   side=(\'right\' if s[\'side\'] == \'left\' else \'left\') if mirror else s[\'side\'])\n                              for i, s in enumerate(stacks)]\n                    sid = f\'{fid}__{role}__m{int(mirror)}\'\n                    records.append(dict(scene_id=sid, role=role, mirror=mirror,\n                                        stacks=actual, features=scene_features(actual)))\n            families.append(dict(family_id=fid, stratum=stratum, phase=phase, scenes=records))\n            used.update(sigs)\n            accepted += 1\n            if accepted == count:\n                break\n        assert accepted == count, f\'Insufficient unused {stratum} geometries: {accepted}/{count}; do not relax exclusions.\'\n    families.sort(key=lambda f: digest([seed, f[\'family_id\']]))\n    validate_families(families, excluded)\n    return families\n\n\ndef validate_families(families, excluded=()):\n    used = set(excluded)\n    assert len({f[\'family_id\'] for f in families}) == len(families)\n    for family in families:\n        base = [s for s in family[\'scenes\'] if not s[\'mirror\']]\n        sigs = {signature(s[\'stacks\']) for s in base}\n        assert len(sigs) == len(base) and not sigs & used\n        used.update(sigs)\n        for scene in family[\'scenes\']:\n            assert scene_features(scene[\'stacks\']) == scene[\'features\']\n            assert len(scene[\'stacks\']) == 4\n            assert len({(s[\'side\'], s[\'distance\']) for s in scene[\'stacks\']}) == 4\n            assert len({s[\'stack_id\'] for s in scene[\'stacks\']}) == 4\n            assert all(1 <= s[\'units\'] <= 6 and 1 <= s[\'distance\'] <= 6 for s in scene[\'stacks\'])\n        if family[\'stratum\'] == \'binding_swap\':\n            a, b = base\n            assert a[\'features\'][\'net_torque\'] * b[\'features\'][\'net_torque\'] < 0\n            for side in (\'left\', \'right\'):\n                aa = [s for s in a[\'stacks\'] if s[\'side\'] == side]\n                bb = [s for s in b[\'stacks\'] if s[\'side\'] == side]\n                assert sorted(s[\'units\'] for s in aa) == sorted(s[\'units\'] for s in bb)\n                assert sorted(s[\'distance\'] for s in aa) == sorted(s[\'distance\'] for s in bb)\n            assert {s[\'stack_id\']: (s[\'side\'], s[\'distance\']) for s in a[\'stacks\']} == {\n                s[\'stack_id\']: (s[\'side\'], s[\'distance\']) for s in b[\'stacks\']}\n\n\ndef mappings_for(family_id):\n    offset = int(digest([family_id, \'mapping\'])[:8], 16) % 3\n    return [tuple(SEMANTICS[(i + k + offset) % 3] for i in range(3)) for k in range(3)]\n\n\ndef public_fields(scene, condition):\n    assert condition in CONDITIONS\n    full = condition in (\'image_full\', \'text_full\')\n    return [dict(stack_id=s[\'stack_id\'], side=s[\'side\'],\n                 weight=s[\'units\'] if full or condition == \'image_weight\' else \'not provided\',\n                 distance=s[\'distance\'] if full or condition == \'image_distance\' else \'not provided\')\n            for s in sorted(scene[\'stacks\'], key=lambda s: s[\'stack_id\'])]\n\n\ndef prompt_for(scene, condition, semantics):\n    options = \'\\n\'.join(f\'{label}. {SEMANTIC_TEXT[meaning]}\' for label, meaning in zip(LABELS, semantics))\n    if condition == \'legacy_image\':\n        parts = [\'Answer this question using only the supplied balance-scale scene.\', RULE_TEXT]\n    else:\n        fields = public_fields(scene, condition)\n        table = \'\\n\'.join(f"{r[\'stack_id\']}: side={r[\'side\']}; weight={r[\'weight\']}; distance={r[\'distance\']}" for r in fields)\n        parts = [\'Answer this question using the supplied scene information.\', RULE_TEXT,\n                 \'Stack IDs identify positions; they do not encode weight or distance. \'\n                 \'Use any supplied image and provided values. A field marked not provided supplies no numeric value.\',\n                 \'Provided stack information:\\n\' + table]\n    parts += [\'Question: Which way will the balance scale tip?\', \'Options:\\n\' + options, REPLY_TEXT]\n    return \'\\n\\n\'.join(parts)\n\n\ndef trial_specs(families):\n    for family in families:\n        for scene in family[\'scenes\']:\n            for mapping_index, semantics in enumerate(mappings_for(family[\'family_id\'])):\n                for condition in CONDITIONS:\n                    eid = scene[\'scene_id\'] + f\'__map{mapping_index}__{condition}\'\n                    yield dict(trial_id=eid, family_id=family[\'family_id\'], stratum=family[\'stratum\'],\n                               scene_id=scene[\'scene_id\'], role=scene[\'role\'], mirror=scene[\'mirror\'],\n                               mapping_index=mapping_index, semantics=list(semantics), condition=condition,\n                               gold_semantic=scene[\'features\'][\'outcome\'],\n                               gold_label=LABELS[semantics.index(scene[\'features\'][\'outcome\'])],\n                               weight_outcome=scene[\'features\'][\'weight_outcome\'])\n\n\ndef render_images(families, image_dir):\n    from visual_torque_toolkit.render import RenderStyle, render_scene\n    from visual_torque_toolkit.world import Scene, Side, Stack\n    image_dir = Path(image_dir)\n    image_dir.mkdir(parents=True, exist_ok=True)\n    lock = image_dir.parent / \'image_manifest.json\'\n    if lock.exists():\n        manifest = read_json(lock)\n        expected = {s[\'scene_id\'] + suffix for f in families for s in f[\'scenes\']\n                    for suffix in (\'__plain.png\', \'__ids.png\')}\n        assert set(manifest) == expected\n        for name, entry in manifest.items():\n            assert file_hash(image_dir / name) == entry[\'sha256\'], \'Frozen image missing/changed: \' + name\n        return manifest\n    style = RenderStyle()\n    # Built-in scalable Pillow font travels with the pinned Pillow package.\n    font = ImageFont.load_default(size=18)\n    manifest = {}\n    for family in families:\n        for record in family[\'scenes\']:\n            scene = Scene(record[\'scene_id\'], tuple(Stack(s[\'stack_id\'], Side(s[\'side\']), s[\'distance\'], s[\'units\'])\n                                                   for s in record[\'stacks\']))\n            plain = image_dir / (record[\'scene_id\'] + \'__plain.png\')\n            render_scene(scene, plain, style)\n            with Image.open(plain) as source:\n                labeled = source.convert(\'RGB\')\n            draw = ImageDraw.Draw(labeled)\n            spacing = int(style.width * .38) / style.ticks_per_side\n            boxes = []\n            for s in scene.stacks:\n                x = round(style.width // 2 + (-1 if s.side is Side.LEFT else 1) * s.distance * spacing)\n                y = style.beam_y + 25\n                box = draw.textbbox((x, y), s.stack_id, font=font, anchor=\'mt\')\n                assert 0 <= box[0] < box[2] < style.width and 0 <= box[1] < box[3] < style.height\n                assert all(box[2] <= b[0] or box[0] >= b[2] or box[3] <= b[1] or box[1] >= b[3] for b in boxes)\n                boxes.append(box)\n                draw.text((x, y), s.stack_id, font=font, anchor=\'mt\', fill=style.ink)\n            labeled.save(image_dir / (record[\'scene_id\'] + \'__ids.png\'))\n            for name in (plain.name, record[\'scene_id\'] + \'__ids.png\'):\n                manifest[name] = dict(sha256=file_hash(image_dir / name), width=style.width, height=style.height)\n    save_json(lock, manifest)\n    return manifest\n\n\ndef generation_audit_ids(families):\n    selected = set()\n    for stratum in STRATA:\n        choices = sorted((f for f in families if f[\'stratum\'] == stratum), key=lambda f: digest([\'generation\', f[\'family_id\']]))[:2]\n        for f in choices:\n            scene = next(s for s in f[\'scenes\'] if s[\'role\'] == \'base\' and not s[\'mirror\'])\n            k = int(digest([\'audit-map\', f[\'family_id\']])[:8], 16) % 3\n            selected.update(scene[\'scene_id\'] + f\'__map{k}__{c}\' for c in CONDITIONS)\n    return selected\n\n\ndef strict_label(text):\n    value = text.strip()\n    return value if value in LABELS else None\n\n\ndef holm(p_values):\n    p = np.asarray(p_values, dtype=float)\n    order = np.argsort(p)\n    adjusted = np.empty(len(p))\n    previous = 0.\n    for rank, i in enumerate(order):\n        previous = max(previous, min(1., p[i] * (len(p) - rank)))\n        adjusted[i] = previous\n    return adjusted\n\n\ndef paired_stats(values, key, replicates=10000):\n    x = np.asarray(values, dtype=float)\n    assert x.ndim == 1 and len(x) and np.isfinite(x).all()\n    rng = np.random.default_rng(int(digest(key)[:16], 16))\n    draws = np.concatenate([rng.choice(x, (min(250, replicates-i), len(x)), replace=True).mean(axis=1)\n                            for i in range(0, replicates, 250)])\n    nonzero = x[x != 0]\n    observed = abs(float(x.sum()))\n    if len(nonzero) <= 16:\n        permuted = np.array([sum(a*b for a, b in zip(signs, nonzero))\n                             for signs in itertools.product((-1., 1.), repeat=len(nonzero))])\n        p = float(np.mean(np.abs(permuted) >= observed - 1e-12))\n    else:\n        extreme = 0\n        for i in range(0, replicates, 250):\n            signs = rng.choice((-1., 1.), size=(min(250, replicates-i), len(nonzero)))\n            extreme += int((np.abs(signs @ nonzero) >= observed - 1e-12).sum())\n        p = (extreme + 1) / (replicates + 1)\n    return dict(n_families=int(len(x)), mean=float(x.mean()),\n                lower=float(np.quantile(draws, .025)), upper=float(np.quantile(draws, .975)), p_value=p)\n\n\ndef analyze_rows(rows, families, outdir, model_key, phase, replicates=10000):\n    outdir = Path(outdir)\n    outdir.mkdir(parents=True, exist_ok=True)\n    df = pd.DataFrame(rows)\n    expected = {s[\'trial_id\'] for s in trial_specs(families)}\n    assert df.trial_id.is_unique and set(df.trial_id) == expected, \'Analysis requires every planned trial.\'\n    df.to_json(outdir / \'trial_results.jsonl\', orient=\'records\', lines=True, force_ascii=False)\n    metrics = [\'correct\', \'gold_margin\', \'predicted_balance\', \'wrong_follows_weight\']\n    fam = df.groupby([\'family_id\', \'stratum\', \'condition\'])[metrics].mean().reset_index()\n    fam.to_csv(outdir / \'family_metrics.csv\', index=False)\n    summaries = []\n    for scope in (\'all\', *STRATA):\n        subset = fam if scope == \'all\' else fam[fam.stratum == scope]\n        for condition, group in subset.groupby(\'condition\'):\n            row = dict(model=model_key, phase=phase, scope=scope, condition=condition,\n                       **paired_stats(group.correct, [model_key, phase, scope, condition], replicates))\n            row.pop(\'p_value\')\n            row.update(gold_margin=float(group.gold_margin.mean()),\n                       balance_answer_rate=float(group.predicted_balance.mean()),\n                       wrong_weight_response_rate=float(group.wrong_follows_weight.mean()))\n            summaries.append(row)\n    summary = pd.DataFrame(summaries)\n    summary.to_csv(outdir / \'accuracy_by_condition.csv\', index=False)\n    pair_rows = []\n    subset = df[df.stratum == \'binding_swap\']\n    for keys, group in subset.groupby([\'family_id\', \'condition\', \'mirror\', \'mapping_index\']):\n        assert set(group.role) == {\'base\', \'swapped\'} and len(group) == 2\n        a, b = [group[group.role == role].iloc[0] for role in (\'base\', \'swapped\')]\n        pair_rows.append(dict(family_id=keys[0], condition=keys[1], mirror=keys[2], mapping_index=keys[3],\n                              both_correct=int(a.correct and b.correct),\n                              answer_changed=int(a.predicted_semantic != b.predicted_semantic)))\n    pairs = pd.DataFrame(pair_rows)\n    pairs.to_csv(outdir / \'binding_pair_trials.csv\', index=False)\n    pair_fam = pairs.groupby([\'family_id\', \'condition\'])[[\'both_correct\', \'answer_changed\']].mean().reset_index()\n    pair_fam.to_csv(outdir / \'binding_pair_families.csv\', index=False)\n    definitions = [\n        (\'weight_support_all\', \'all\', [(\'image_weight\', 1), (\'image\', -1)], \'correct\'),\n        (\'distance_support_all\', \'all\', [(\'image_distance\', 1), (\'image\', -1)], \'correct\'),\n        (\'full_support_all\', \'all\', [(\'image_full\', 1), (\'image\', -1)], \'correct\'),\n        (\'joint_support_interaction\', \'all\', [(\'image_full\', 1), (\'image_weight\', -1), (\'image_distance\', -1), (\'image\', 1)], \'correct\'),\n        (\'distance_equal_weight\', \'equal_weight_distance\', [(\'image_distance\', 1), (\'image\', -1)], \'correct\'),\n        (\'full_weight_conflict\', \'weight_conflict\', [(\'image_full\', 1), (\'image\', -1)], \'correct\'),\n        (\'full_binding_pair\', \'binding_swap\', [(\'image_full\', 1), (\'image\', -1)], \'both_correct\'),\n    ]\n    comparisons = []\n    for name, scope, coefficients, metric in definitions:\n        source = pair_fam if metric == \'both_correct\' else (fam if scope == \'all\' else fam[fam.stratum == scope])\n        pivot = source.pivot(index=\'family_id\', columns=\'condition\', values=metric)\n        delta = sum(coef * pivot[condition] for condition, coef in coefficients)\n        comparisons.append(dict(model=model_key, phase=phase, comparison=name, metric=metric, scope=scope,\n                                **paired_stats(delta.to_numpy(), [model_key, phase, name], replicates)))\n    tests = pd.DataFrame(comparisons)\n    tests[\'holm_within_model\'] = holm(tests.p_value)\n    tests[\'inference_status\'] = \'development_descriptive_only\' if phase == \'development\' else \'pending_joint_14_test_correction\'\n    tests.to_csv(outdir / \'primary_comparisons.csv\', index=False)\n    event_rows = []\n    keys = [\'family_id\', \'scene_id\', \'role\', \'mirror\', \'mapping_index\']\n    base = df[df.condition == \'image\'][keys + [\'correct\']].rename(columns={\'correct\': \'baseline_correct\'})\n    for c in CONDITIONS:\n        joined = df[df.condition == c].merge(base, on=keys, validate=\'one_to_one\')\n        joined[\'repaired\'] = (joined.baseline_correct == 0) & (joined.correct == 1)\n        joined[\'harmed\'] = (joined.baseline_correct == 1) & (joined.correct == 0)\n        joined[\'original_error\'] = joined.baseline_correct == 0\n        joined[\'original_correct\'] = joined.baseline_correct == 1\n        for scope in (\'all\', *STRATA):\n            g = joined if scope == \'all\' else joined[joined.stratum == scope]\n            event_rows.append(dict(condition=c, scope=scope, n_trials=len(g),\n                                   repairs=int(g.repaired.sum()), harms=int(g.harmed.sum()),\n                                   original_errors=int(g.original_error.sum()), original_correct=int(g.original_correct.sum()),\n                                   repairs_per_original_error=float(g.repaired.sum()/g.original_error.sum()) if g.original_error.sum() else None,\n                                   harms_per_original_correct=float(g.harmed.sum()/g.original_correct.sum()) if g.original_correct.sum() else None))\n    pd.DataFrame(event_rows).to_csv(outdir / \'repair_harm_counts.csv\', index=False)\n    audits = df[df.generation_audited == True]\n    audit_summary = []\n    for condition, group in audits.groupby(\'condition\'):\n        valid = group.generated_label.isin(LABELS)\n        agreed = valid & (group.generated_label == group.predicted_label)\n        audit_summary.append(dict(condition=condition, n=len(group), valid=int(valid.sum()),\n                                  agrees=int(agreed.sum()), agreement_all=float(agreed.mean()),\n                                  agreement_valid=float(agreed.sum()/valid.sum()) if valid.sum() else None))\n    pd.DataFrame(audit_summary).to_csv(outdir / \'generation_scoring_agreement.csv\', index=False)\n    return summary, tests\n',"<embedded-parent>","exec"),_parent.__dict__)
exec(compile('"""Frozen, family-disjoint visual-information and wording validation."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw\nimport visual_information_support_core as base\nfrom visual_information_support_core import (\n    digest, file_hash, read_json, save_json, freeze, MODEL_SPECS, LABELS,\n    SEMANTIC_TEXT, CONDITIONS, STRATA, strict_label,\n)\n\nSCHEMA=\'frozen-visual-validation-v1\'\nPHASE=\'frozen_validation\'\nN_FAMILIES=96\nFROZEN_SEED=2026091803\nWORDINGS=(\'original\',\'neutral_question\')\nPRIMARY_DEFINITIONS=[\n    (\'weight_support_all\',\'all\',[(\'image_weight\',1),(\'image\',-1)],\'correct\'),\n    (\'distance_support_all\',\'all\',[(\'image_distance\',1),(\'image\',-1)],\'correct\'),\n    (\'full_support_all\',\'all\',[(\'image_full\',1),(\'image\',-1)],\'correct\'),\n    (\'joint_support_interaction\',\'all\',[(\'image_full\',1),(\'image_weight\',-1),(\'image_distance\',-1),(\'image\',1)],\'correct\'),\n    (\'distance_equal_weight\',\'equal_weight_distance\',[(\'image_distance\',1),(\'image\',-1)],\'correct\'),\n    (\'full_weight_conflict\',\'weight_conflict\',[(\'image_full\',1),(\'image\',-1)],\'correct\'),\n    (\'full_binding_pair\',\'binding_swap\',[(\'image_full\',1),(\'image\',-1)],\'both_correct\'),\n]\n# These four secondary comparisons address the observed wording trade-off.\nWORDING_DEFINITIONS=[\n    (\'image_accuracy_all\',\'all\',\'image\',\'correct\'),\n    (\'image_true_balance\',\'balanced\',\'image\',\'correct\'),\n    (\'image_false_balance\',\'nonbalanced\',\'image\',\'predicted_balance\'),\n    (\'full_binding_pair\',\'binding_swap\',\'image_full\',\'both_correct\'),\n]\n\n\ndef validation_prompt(scene,condition,semantics,wording):\n    assert wording in WORDINGS\n    prompt=base.prompt_for(scene,condition,semantics)\n    if wording==\'neutral_question\':\n        label=LABELS[semantics.index(\'balanced\')]\n        prompt=prompt.replace(f\'{label}. equal\',f\'{label}. balanced (neither side tips)\')\n        prompt=prompt.replace(\'Question: Which way will the balance scale tip?\',\n                              \'Question: What happens to the balance scale?\')\n    return prompt\n\n\ndef freeze_sources(run_root,source_paths,expected_versions,extra_files=()):\n    """Read four named ZIPs once; resume verifies their whole-file hashes only."""\n    run_root=Path(run_root)\n    lock=run_root/\'historical_exclusion.json\'\n    resolved={kind:{key:str(Path(p).resolve()) for key,p in paths.items()} for kind,paths in source_paths.items()}\n    extra=[str(Path(p).resolve()) for p in extra_files]\n    if lock.is_file():\n        old=read_json(lock)\n        assert old[\'source_paths\']==resolved and old[\'extra_files\']==extra,\'Source list changed; do not mix validation protocols.\'\n        assert old[\'expected_versions\']==expected_versions\n        for path,h in old[\'file_hashes\'].items():\n            assert file_hash(path)==h,\'Frozen source file changed: \'+path\n        return old,base.geometries_in(old)\n    excluded=set();evidence={};hashes={};shared_identities={}\n    for kind,paths in resolved.items():\n        phase=\'development\' if kind==\'development\' else \'diagnostic\'\n        evidence[kind]={}\n        for key,path in paths.items():\n            assert Path(path).is_file(),f\'Missing {kind} output for {key}: {path}. Set SOURCE_ZIPS to its exact path.\'\n            h=file_hash(path);hashes[path]=h\n            with zipfile.ZipFile(path) as z:\n                manifest=json.loads(z.read(\'export_manifest.json\'))\n                assert manifest[\'status\']==\'complete\' and manifest[\'model\']==key and manifest[\'phase\']==phase\n                assert set(z.namelist())==set(manifest[\'files\'])|{\'export_manifest.json\'}\n                for name,value in manifest[\'files\'].items():\n                    assert hashlib.sha256(z.read(name)).hexdigest()==value,\'ZIP manifest mismatch: \'+name\n                prefix=f\'models/{key}/{phase}/\'\n                complete=json.loads(z.read(prefix+\'completion.json\'))\n                runtime=json.loads(z.read(prefix+\'runtime_identity.json\'))\n                assert complete[\'status\']==\'complete\'\n                assert runtime[\'implementation\']==complete[\'implementation\']==expected_versions[kind]\n                assert runtime[\'model_metadata\'][\'resolved_revision\']==MODEL_SPECS[key][\'revision\']\n                assert runtime[\'dtype\']==\'bfloat16\' and runtime[\'thinking\'] is False\n                assert complete[\'runtime_sha256\']==hashlib.sha256(z.read(prefix+\'runtime_identity.json\')).hexdigest()\n                support_name=f\'shared/{phase}/support.json\'\n                inputs_name=f\'shared/{phase}/inputs.json\'\n                support=json.loads(z.read(support_name))\n                support_hash=hashlib.sha256(z.read(support_name)).hexdigest()\n                inputs_hash=hashlib.sha256(z.read(inputs_name)).hexdigest()\n                assert runtime[\'support_sha256\']==complete[\'support_sha256\']==support_hash\n                assert runtime[\'inputs_sha256\']==inputs_hash\n                identity=(support_hash,inputs_hash)\n                if kind in shared_identities:\n                    assert identity==shared_identities[kind],\'The models did not share prior inputs.\'\n                else:\n                    shared_identities[kind]=identity\n                excluded.update(base.geometries_in(support))\n                if kind==\'development\':\n                    excluded.update(base.geometries_in(json.loads(z.read(\'historical_exclusion.json\'))))\n                    assert complete[\'n_scored\']==1440\n                    lineage=None\n                else:\n                    assert complete[\'n_scored\']==600\n                    snapshots=json.loads(z.read(\'shared/diagnostic/source_snapshots.json\'))\n                    lineage={k:v[\'zip_sha256\'] for k,v in snapshots.items()}\n                evidence[kind][key]=dict(zip_name=Path(path).name,zip_sha256=h,\n                    support_sha256=support_hash,inputs_sha256=inputs_hash,runtime=runtime,\n                    development_lineage=lineage)\n    assert set(evidence)=={\'development\',\'diagnostic\'}\n    for d in evidence[\'diagnostic\'].values():\n        assert d[\'development_lineage\']=={key:x[\'zip_sha256\'] for key,x in evidence[\'development\'].items()}\n    for name in extra:\n        path=Path(name)\n        assert path.is_file(),\'Extra exclusions must name a JSON or JSONL manifest: \'+name\n        hashes[name]=file_hash(path)\n        if path.suffix==\'.jsonl\':\n            with path.open(encoding=\'utf-8-sig\') as stream:\n                for line in stream:\n                    if line.strip():\n                        excluded.update(base.geometries_in(json.loads(line)))\n        else:\n            excluded.update(base.geometries_in(json.loads(path.read_text(encoding=\'utf-8-sig\'))))\n    snapshot=dict(schema=SCHEMA,expected_versions=expected_versions,source_paths=resolved,extra_files=extra,\n        file_hashes=hashes,evidence=evidence,signatures=[[list(t) for t in s] for s in sorted(excluded)],\n        scope=\'Known development history, development and diagnostic geometries, and explicit extra manifests only; mirrors canonicalized.\')\n    freeze(lock,snapshot)\n    return snapshot,excluded\n\n\ndef make_inputs(families):\n    generation_families={}\n    for group in STRATA:\n        fs=sorted((f for f in families if f[\'stratum\']==group),key=lambda f:digest([\'generation\',f[\'family_id\']]))\n        generation_families[group]={f[\'family_id\'] for f in fs[:2]}\n    inputs=[]\n    scenes={s[\'scene_id\']:s for f in families for s in f[\'scenes\']}\n    for spec in base.trial_specs(families):\n        scene=scenes[spec[\'scene_id\']]\n        audit_map=int(digest([\'audit-map\',spec[\'family_id\']])[:8],16)%3\n        audited=spec[\'family_id\'] in generation_families[spec[\'stratum\']] and spec[\'role\']==\'base\' and spec[\'mapping_index\']==audit_map\n        for wording in WORDINGS:\n            prompt=validation_prompt(scene,spec[\'condition\'],spec[\'semantics\'],wording)\n            name=None if spec[\'condition\']==\'text_full\' else spec[\'scene_id\']+(\'__plain.png\' if spec[\'condition\']==\'legacy_image\' else \'__ids.png\')\n            inputs.append(dict(spec,trial_id=spec[\'trial_id\']+\'__\'+wording,parent_trial_id=spec[\'trial_id\'],\n                wording=wording,prompt=prompt,prompt_sha256=hashlib.sha256(prompt.encode()).hexdigest(),\n                image_filename=name,generation_audit=audited))\n    assert len(inputs)==len({x[\'trial_id\'] for x in inputs})\n    return inputs\n\n\ndef prepare_validation(shared_root,source_snapshot,excluded):\n    shared_root=Path(shared_root)\n    path=shared_root/\'support.json\'\n    phase_hash=file_hash(shared_root/\'phase_protocol.json\')\n    source_hash=digest(source_snapshot)\n    if path.exists():\n        support=read_json(path)\n        assert support[\'source_snapshot_sha256\']==source_hash and support[\'phase_protocol_sha256\']==phase_hash\n        assert support[\'n_families\']==N_FAMILIES and support[\'seed\']==FROZEN_SEED\n        families=support[\'families\']\n    else:\n        families=base.generate_families(N_FAMILIES,FROZEN_SEED,excluded,PHASE)\n        support=dict(schema=SCHEMA,phase=PHASE,n_families=N_FAMILIES,seed=FROZEN_SEED,families=families,\n            source_snapshot_sha256=source_hash,phase_protocol_sha256=phase_hash)\n        freeze(path,support)\n    assert len(families)==N_FAMILIES\n    assert {s:sum(f[\'stratum\']==s for f in families) for s in STRATA}==base.allocation(N_FAMILIES)\n    base.validate_families(families,excluded)\n    images=base.render_images(families,shared_root/\'images\')\n    inputs=make_inputs(families)\n    for inp in inputs:\n        inp[\'image_sha256\']=images[inp[\'image_filename\']][\'sha256\'] if inp[\'image_filename\'] else None\n    assert len(inputs)==8640 and sum(x[\'generation_audit\'] for x in inputs)==240\n    freeze(shared_root/\'inputs.json\',inputs)\n    freeze(shared_root/\'generation_audit_plan.json\',[x[\'trial_id\'] for x in inputs if x[\'generation_audit\']])\n    freeze(shared_root/\'geometry_checks.json\',dict(n_families=N_FAMILIES,n_scene_presentations=240,\n        n_mirror_canonical_geometries=len(base.geometries_in(families)),\n        excluded_geometries=len(excluded),overlap_with_recorded_history=0,\n        strata=base.allocation(N_FAMILIES),scope=source_snapshot[\'scope\']))\n    return families,inputs\n\n\ndef analyze_validation(rows,inputs,families,outdir,model_key,replicates=10000):\n    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n    df=pd.DataFrame(rows)\n    expected={x[\'trial_id\']:x for x in inputs}\n    assert df.trial_id.is_unique and set(df.trial_id)==set(expected),\'Complete all planned inputs before final analysis.\'\n    for r in rows:\n        inp=expected[r[\'trial_id\']]\n        for k in (\'family_id\',\'scene_id\',\'condition\',\'wording\',\'gold_label\',\'gold_semantic\',\'mapping_index\',\'mirror\',\'prompt_sha256\'):\n            assert r[k]==inp[k]\n        vals=[r[\'candidate_logprobs\'][l] for l in LABELS]\n        assert np.isfinite(vals).all()\n        label=LABELS[int(np.argmax(vals))]\n        assert r[\'predicted_label\']==label and r[\'predicted_semantic\']==r[\'semantics\'][LABELS.index(label)]\n        assert r[\'correct\']==int(label==r[\'gold_label\'])\n        assert r[\'generation_audited\']==inp[\'generation_audit\']\n    df.to_json(outdir/\'trial_results.jsonl\',orient=\'records\',lines=True,force_ascii=False)\n    for side in (\'left\',\'right\',\'balanced\'):\n        df[\'answer_\'+side]=(df.predicted_semantic==side).astype(int)\n    for letter in LABELS:\n        df[\'label_\'+letter]=(df.predicted_label==letter).astype(int)\n    metrics=[\'correct\',\'gold_margin\',\'predicted_balance\',\'wrong_follows_weight\',\'top_score_tie\',\n             \'answer_left\',\'answer_right\',\'answer_balanced\',\'label_A\',\'label_B\',\'label_C\']\n    fam=df.groupby([\'family_id\',\'stratum\',\'condition\',\'wording\'])[metrics].mean().reset_index()\n    fam.to_csv(outdir/\'family_metrics.csv\',index=False)\n    summaries=[]\n    for scope in (\'all\',*STRATA):\n        subset=fam if scope==\'all\' else fam[fam.stratum==scope]\n        for (condition,wording),g in subset.groupby([\'condition\',\'wording\']):\n            stats=base.paired_stats(g.correct,[model_key,PHASE,scope,condition,wording],replicates)\n            stats.pop(\'p_value\')\n            summaries.append(dict(model=model_key,scope=scope,condition=condition,wording=wording,\n                **stats,**{k:float(g[k].mean()) for k in metrics if k!=\'correct\'}))\n    summary=pd.DataFrame(summaries)\n    summary.to_csv(outdir/\'accuracy_and_answer_bias.csv\',index=False)\n    pairs=[]\n    for k,g in df[df.stratum==\'binding_swap\'].groupby([\'family_id\',\'condition\',\'wording\',\'mirror\',\'mapping_index\']):\n        assert len(g)==2 and set(g.role)=={\'base\',\'swapped\'}\n        a=g[g.role==\'base\'].iloc[0];b=g[g.role==\'swapped\'].iloc[0]\n        pairs.append(dict(zip([\'family_id\',\'condition\',\'wording\',\'mirror\',\'mapping_index\'],k),\n            both_correct=int(a.correct and b.correct),answer_changed=int(a.predicted_semantic!=b.predicted_semantic)))\n    pair_df=pd.DataFrame(pairs);pair_df.to_csv(outdir/\'binding_pair_trials.csv\',index=False)\n    pair_fam=pair_df.groupby([\'family_id\',\'condition\',\'wording\'])[[\'both_correct\',\'answer_changed\']].mean().reset_index()\n    pair_fam.to_csv(outdir/\'binding_pair_families.csv\',index=False)\n    effects=[]\n    for wording in WORDINGS:\n        for name,scope,coefficients,metric in PRIMARY_DEFINITIONS:\n            source=pair_fam[pair_fam.wording==wording] if metric==\'both_correct\' else fam[fam.wording==wording]\n            if metric!=\'both_correct\' and scope!=\'all\':\n                source=source[source.stratum==scope]\n            pivot=source.pivot(index=\'family_id\',columns=\'condition\',values=metric)\n            delta=sum(coef*pivot[condition] for condition,coef in coefficients)\n            stats=base.paired_stats(delta.to_numpy(),[model_key,PHASE,name,wording],replicates)\n            row=dict(model=model_key,wording=wording,comparison=name,scope=scope,metric=metric,**stats)\n            if wording!=\'original\':\n                row.pop(\'p_value\')\n            effects.append(row)\n    effect_df=pd.DataFrame(effects)\n    primary=effect_df[effect_df.wording==\'original\'].copy()\n    primary[\'holm_within_model_7\']=base.holm(primary.p_value)\n    primary[\'inference_status\']=\'pending_two_model_primary_14_correction\'\n    primary.to_csv(outdir/\'primary_original_comparisons.csv\',index=False)\n    neutral=effect_df[effect_df.wording!=\'original\'].drop(columns=\'p_value\').copy()\n    neutral[\'inference_status\']=\'descriptive_paired_intervals_no_significance_claim\'\n    neutral.to_csv(outdir/\'neutral_information_effects_descriptive.csv\',index=False)\n    secondary=[]\n    for name,scope,condition,metric in WORDING_DEFINITIONS:\n        source=pair_fam if metric==\'both_correct\' else fam\n        if metric!=\'both_correct\' and scope!=\'all\':\n            source=source[source.stratum!=\'balanced\'] if scope==\'nonbalanced\' else source[source.stratum==scope]\n        pivot=source[source.condition==condition].pivot(index=\'family_id\',columns=\'wording\',values=metric)\n        stats=base.paired_stats((pivot.neutral_question-pivot.original).to_numpy(),[model_key,PHASE,\'wording\',name],replicates)\n        secondary.append(dict(model=model_key,comparison=name,scope=scope,condition=condition,metric=metric,\n                              contrast=\'neutral_question minus original\',**stats))\n    secondary=pd.DataFrame(secondary)\n    secondary[\'holm_within_model_4\']=base.holm(secondary.p_value)\n    secondary[\'inference_status\']=\'pending_two_model_secondary_8_correction\'\n    secondary.to_csv(outdir/\'secondary_wording_comparisons.csv\',index=False)\n    # Every condition\'s paired repairs and harms, including denominator-zero cases.\n    events=[]\n    index=[\'family_id\',\'scene_id\',\'role\',\'mirror\',\'mapping_index\']\n    def event_summary(a,b,metadata):\n        joined=a[index+[\'stratum\',\'correct\']].rename(columns={\'correct\':\'before\'}).merge(\n            b[index+[\'correct\']],on=index,validate=\'one_to_one\').rename(columns={\'correct\':\'after\'})\n        for scope in (\'all\',*STRATA):\n            g=joined if scope==\'all\' else joined[joined.stratum==scope]\n            errors=int((g.before==0).sum());correct=int((g.before==1).sum())\n            repairs=int(((g.before==0)&(g.after==1)).sum());harms=int(((g.before==1)&(g.after==0)).sum())\n            events.append(dict(**metadata,scope=scope,n_trials=len(g),repairs=repairs,harms=harms,\n                original_errors=errors,original_correct=correct,\n                repairs_per_original_error=repairs/errors if errors else None,\n                harms_per_original_correct=harms/correct if correct else None))\n    for wording in WORDINGS:\n        source=df[df.wording==wording]\n        for condition in CONDITIONS:\n            event_summary(source[source.condition==\'image\'],source[source.condition==condition],\n                          dict(comparison_type=\'information_support\',condition=condition,wording=wording))\n    for condition in CONDITIONS:\n        source=df[df.condition==condition]\n        event_summary(source[source.wording==\'original\'],source[source.wording==\'neutral_question\'],\n                      dict(comparison_type=\'wording\',condition=condition,wording=\'neutral_minus_original\'))\n    pd.DataFrame(events).to_csv(outdir/\'repairs_and_harms.csv\',index=False)\n    balance=[]\n    for (condition,wording),g in df.groupby([\'condition\',\'wording\']):\n        for truth,sub in [(\'true_balance\',g[g.gold_semantic==\'balanced\']),(\'should_tip\',g[g.gold_semantic!=\'balanced\'])]:\n            family=sub.groupby(\'family_id\').predicted_balance.mean()\n            balance.append(dict(condition=condition,wording=wording,truth=truth,n_families=len(family),\n                family_mean_balance_response=float(family.mean()),n_trials=len(sub),balanced_responses=int(sub.predicted_balance.sum())))\n    pd.DataFrame(balance).to_csv(outdir/\'true_and_false_balance.csv\',index=False)\n    mirror=[];flip={\'left\':\'right\',\'right\':\'left\',\'balanced\':\'balanced\'}\n    for k,g in df.groupby([\'family_id\',\'stratum\',\'role\',\'condition\',\'wording\',\'mapping_index\']):\n        assert len(g)==2 and set(g.mirror)=={False,True}\n        a=g[g.mirror==False].iloc[0];b=g[g.mirror==True].iloc[0]\n        mirror.append(dict(zip([\'family_id\',\'stratum\',\'role\',\'condition\',\'wording\',\'mapping_index\'],k),\n            both_correct=int(a.correct and b.correct),semantic_equivariant=int(flip[a.predicted_semantic]==b.predicted_semantic)))\n    pd.DataFrame(mirror).to_csv(outdir/\'mirror_pairs.csv\',index=False)\n    maps=[]\n    for k,g in df.groupby([\'family_id\',\'stratum\',\'scene_id\',\'condition\',\'wording\']):\n        assert len(g)==3 and g.mapping_index.nunique()==3\n        maps.append(dict(zip([\'family_id\',\'stratum\',\'scene_id\',\'condition\',\'wording\'],k),\n            semantic_invariant=int(g.predicted_semantic.nunique()==1),all_correct=int(g.correct.min())))\n    pd.DataFrame(maps).to_csv(outdir/\'mapping_invariance.csv\',index=False)\n    audits=[]\n    for (condition,wording),g in df[df.generation_audited].groupby([\'condition\',\'wording\']):\n        valid=g.generated_label.isin(LABELS);agree=valid&(g.generated_label==g.predicted_label)\n        audits.append(dict(condition=condition,wording=wording,n=len(g),valid=int(valid.sum()),agrees=int(agree.sum()),\n            agreement_all=float(agree.mean()),agreement_valid=float(agree.sum()/valid.sum()) if valid.sum() else None))\n    pd.DataFrame(audits).to_csv(outdir/\'generation_agreement.csv\',index=False)\n    # Constant-side baseline under the pre-fixed strata mixture, not uniform chance.\n    gold=df[(df.condition==\'image\')&(df.wording==\'original\')]\n    save_json(outdir/\'simple_policy_baselines.json\',{side:float(gold.groupby(\'family_id\').gold_semantic.apply(lambda x:float((x==side).mean())).mean()) for side in (\'left\',\'right\',\'balanced\')})\n    return summary,primary,secondary\n',"<embedded-frozen-validation>","exec"),globals())
IMPLEMENTATION_SHA256='e84045fd42cd1f058593cdbd45e07115e0339b491ea91c315ffdee99be1e75d4'
EXPECTED_SOURCE_VERSIONS={'development': '2e4dd2772d1131a18a47f3a23766fdb5985281134b88d03045a8c0261d0c3e6e', 'diagnostic': '5a4cc32e046dbbcc6cd752ffcb9499f79db32a48adb86e2531a8cdc88b21c154'}


In [ ]:
%%cell_progress 5/13 Freeze validation protocol
assert MODEL_KEY in MODEL_SPECS
for value in (RUN_ID,SOURCE_DEVELOPMENT_RUN_ID,SOURCE_DIAGNOSTIC_RUN_ID):
    assert value and all(c.isalnum() or c in '_-' for c in value)
SPEC=MODEL_SPECS[MODEL_KEY]
SEED=FROZEN_SEED
RUN_ROOT=PROJECT_ROOT/'results/frozen_visual_validation_v1'/RUN_ID
SHARED_ROOT=RUN_ROOT/'shared'/PHASE
PHASE_ROOT=RUN_ROOT/'models'/MODEL_KEY/PHASE
ANALYSIS_DIR=PHASE_ROOT/'analysis'
IMAGE_DIR=SHARED_ROOT/'images'
SHARED_ROOT.mkdir(parents=True,exist_ok=True);ANALYSIS_DIR.mkdir(parents=True,exist_ok=True)
SOURCE_PATHS={kind:{} for kind in ('development','diagnostic')}
for key in MODEL_SPECS:
    SOURCE_PATHS['development'][key]=Path(SOURCE_ZIPS['development'][key]) if SOURCE_ZIPS['development'][key] else (
        PROJECT_ROOT/'results/visual_information_support_v1'/SOURCE_DEVELOPMENT_RUN_ID/
        f'VisualTorque_VisualInformationSupport_{key}_development_Output.zip')
    SOURCE_PATHS['diagnostic'][key]=Path(SOURCE_ZIPS['diagnostic'][key]) if SOURCE_ZIPS['diagnostic'][key] else (
        PROJECT_ROOT/'results/direction_balance_diagnostic_v1'/SOURCE_DIAGNOSTIC_RUN_ID/
        f'VisualTorque_Direction_Balance_Diagnostic_{key}_Output.zip')
EXCLUSION_PATHS=[Path(p) for p in EXTRA_EXCLUSION_FILES]
older_validation=PROJECT_ROOT/'results/visual_information_support_v1'/SOURCE_DEVELOPMENT_RUN_ID/'shared/validation/support.json'
if older_validation.is_file() and older_validation not in EXCLUSION_PATHS:
    EXCLUSION_PATHS.append(older_validation)
freeze(RUN_ROOT/'protocol.json',dict(schema=SCHEMA,implementation=IMPLEMENTATION_SHA256,
    source_versions=EXPECTED_SOURCE_VERSIONS,models=MODEL_SPECS,n_families=N_FAMILIES,seed=FROZEN_SEED,
    allocation=base.allocation(N_FAMILIES),conditions=list(CONDITIONS),wordings=list(WORDINGS),
    primary_hypotheses=json.loads(json.dumps(PRIMARY_DEFINITIONS)),
    secondary_wording_hypotheses=json.loads(json.dumps(WORDING_DEFINITIONS)),
    primary='Seven original-wording comparisons per model; Holm correction jointly across 14 tests.',
    secondary='Four pre-fixed neutral-minus-original comparisons per model; Holm across 8 tests in a separate secondary family.',
    neutral_information_effects='Paired confidence intervals only; no separate significance claims.',
    inference='Native BF16 batch1, thinking disabled; frozen parent scoring; 8640 scores + 240 short-generation checks/model.',
    geometry='Four stacks, two per side, integer weights/distances 1..6; mirrors and swap partners grouped by physical family.',
    novelty_scope='Unseen relative to recorded geometries; no claim of universal training-data novelty.',
    wording='Neutral variant jointly changes equal to balanced (neither side tips), and tip question to What happens to the balance scale?.',
    interpretation='Treat wording as a pre-fixed package, not a single-word causal ablation or a chosen better prompt.',
    aggregation='Family macro-average; no independent counting of maps, mirrors or swap partners.',
    no_adaptation='Do not tune inputs, change sample size, stop for significance, or choose a model/prompt from validation outcomes.'))
freeze(SHARED_ROOT/'phase_protocol.json',dict(phase=PHASE,n_families=N_FAMILIES,seed=FROZEN_SEED,
    protocol_sha256=file_hash(RUN_ROOT/'protocol.json')))
print('Model:',MODEL_KEY,'| Frozen validation output:',PHASE_ROOT)


In [ ]:
%%cell_progress 6/13 Load scoring and analysis helpers
"""Validation-specific inference glue; parent scoring and checkpoint logic are reused."""
import gc
import shutil
import time
from datetime import datetime, timezone
from tqdm.auto import tqdm
from visual_torque_toolkit.behavior.data import BehaviorExample
from visual_torque_toolkit.behavior.schema import PromptCondition


def make_example(spec):
    inp=INPUT_BY_ID[spec['trial_id']];scene=SCENES[inp['scene_id']]
    path=IMAGE_DIR/inp['image_filename'] if inp['image_filename'] else None
    options=tuple('balanced (neither side tips)' if s=='balanced' and inp['wording']=='neutral_question'
                  else SEMANTIC_TEXT[s] for s in inp['semantics'])
    return BehaviorExample(example_id=inp['trial_id'],question_id=inp['trial_id'],scene_id=inp['scene_id'],
        condition=PromptCondition.SYMBOLIC_RULE if path is None else PromptCondition.IMAGE_RULE,
        question_kind='outcome',variant=inp['condition']+'__'+inp['wording'],split=PHASE,
        problem_type=inp['stratum'],outcome=inp['gold_semantic'],net_torque=scene['features']['net_torque'],
        prompt=inp['prompt'],image_path=path,labels=LABELS,option_text=options,option_semantics=tuple(inp['semantics']),
        gold_label=inp['gold_label'],gold_semantic=inp['gold_semantic'])


def init_validation_runtime():
    init_runtime()
    prior=SOURCE_SNAPSHOT['evidence']['diagnostic'][MODEL_KEY]['runtime']
    fields=('model_metadata','gpu','torch','transformers','pillow','cuda','numpy','pandas',
        'dtype','batch_size','thinking','toolkit','processor_config','image_processor_config',
        'chat_template','tokenizer_chat_template')
    differences={name:dict(identical=RUNTIME_IDENTITY.get(name)==prior.get(name)) for name in fields}
    freeze(PHASE_ROOT/'source_environment_comparison.json',differences)
    changed=[k for k,v in differences.items() if not v['identical']]
    print('Environment changes relative to the diagnostic archive:',changed or 'none')
    critical={'transformers','pillow','dtype','batch_size','thinking','toolkit','processor_config',
              'image_processor_config','chat_template','tokenizer_chat_template'}
    assert not critical.intersection(changed), 'Restore the recorded preprocessing/scoring configuration before validation: '+str(sorted(critical.intersection(changed)))


def show_validation_inputs():
    examples=[]
    for group in STRATA:
        f=next(f for f in FAMILIES if f['stratum']==group)
        s=next(s for s in f['scenes'] if s['role']=='base' and not s['mirror'])
        with Image.open(IMAGE_DIR/(s['scene_id']+'__ids.png')) as im:
            examples.append((group,im.convert('RGB').resize((448,224))))
    sheet=Image.new('RGB',(896,750),'white');draw=ImageDraw.Draw(sheet)
    for i,(label,im) in enumerate(examples):
        x,y=(i%2)*448,(i//2)*250
        sheet.paste(im,(x,y+22));draw.text((x+8,y+4),label,fill='black')
    sheet.save(SHARED_ROOT/'scene_preview.png');display(sheet)
    print('Frozen: 96 physical families; 240 scene presentations; 8,640 scores and 240 short generation checks per model.')
    print('Known excluded mirror-canonical geometries:',len(EXCLUDED))
    print('Original wording is primary. Neutral wording is a fixed parallel comparison, not selected by validation accuracy.')
    print('Family-by-family checkpointing is enabled. Re-run from the top after an interruption.')


def finish_validation(rows):
    summary,primary,secondary=analyze_validation(rows,INPUTS,FAMILIES,ANALYSIS_DIR,MODEL_KEY)
    assert len(rows)==8640 and sum(r['generation_audited'] for r in rows)==240
    display(summary[summary.scope=='all'][['condition','wording','n_families','mean','lower','upper','answer_left','answer_balanced']].round(4))
    display(primary.round(4));display(secondary.round(4))
    agreement=pd.read_csv(ANALYSIS_DIR/'generation_agreement.csv');display(agreement)
    save_json(ANALYSIS_DIR/'measurement_checks.json',dict(
        invalid_generations=int((agreement.n-agreement.valid).sum()),
        disagreements_among_valid=int((agreement.valid-agreement.agrees).sum()),
        checked_generations=int(agreement.n.sum()),
        meaning='Checks describe scoring/short-generation agreement; no post-hoc input exclusions or protocol changes.'))
    save_json(PHASE_ROOT/'completion.json',dict(status='complete',phase=PHASE,model=MODEL_KEY,n_families=96,
        n_scored=8640,n_generation_checks=240,implementation=IMPLEMENTATION_SHA256,
        support_sha256=file_hash(SHARED_ROOT/'support.json'),inputs_sha256=file_hash(SHARED_ROOT/'inputs.json'),
        protocol_sha256=file_hash(RUN_ROOT/'protocol.json'),runtime_sha256=file_hash(PHASE_ROOT/'runtime_identity.json'),
        family_files={p.name:file_hash(p) for p in sorted((PHASE_ROOT/'families').glob('*.json'))},
        analysis_files={p.name:file_hash(p) for p in sorted(ANALYSIS_DIR.glob('*')) if p.is_file() and not p.name.startswith('cross_model')},
        completed_at_utc=now_utc(),scope='Family-disjoint validation under the frozen, previously examined direct-answer protocol.'))
    print('This model is complete. Keep the planned sample size and protocol unchanged; run the other model for joint correction.')
    return summary


def combine_validation_models():
    primary=[];secondary=[];summaries=[]
    for key in MODEL_SPECS:
        root=RUN_ROOT/'models'/key/PHASE
        if not (root/'completion.json').exists():
            print('Joint analysis pending:',key)
            return
        complete=read_json(root/'completion.json')
        assert complete['status']=='complete' and complete['implementation']==IMPLEMENTATION_SHA256
        assert complete['inputs_sha256']==file_hash(SHARED_ROOT/'inputs.json')
        assert complete['protocol_sha256']==file_hash(RUN_ROOT/'protocol.json')
        for name,h in complete['family_files'].items():
            assert file_hash(root/'families'/name)==h,'Recorded family results changed.'
        for name,h in complete['analysis_files'].items():
            assert file_hash(root/'analysis'/name)==h,'Recorded analysis changed.'
        a=pd.read_csv(root/'analysis/primary_original_comparisons.csv')
        b=pd.read_csv(root/'analysis/secondary_wording_comparisons.csv')
        assert len(a)==7 and len(b)==4
        primary.append(a);secondary.append(b)
        summaries.append(pd.read_csv(root/'analysis/accuracy_and_answer_bias.csv'))
    p=pd.concat(primary,ignore_index=True);s=pd.concat(secondary,ignore_index=True)
    p['holm_primary_across_models_14']=base.holm(p.p_value)
    p['inference_status']='primary_family_holm_14_tests'
    s['holm_secondary_across_models_8']=base.holm(s.p_value)
    s['inference_status']='secondary_family_holm_8_tests_separate_from_primary'
    p.to_csv(ANALYSIS_DIR/'cross_model_primary_original.csv',index=False)
    s.to_csv(ANALYSIS_DIR/'cross_model_secondary_wording.csv',index=False)
    pd.concat(summaries,ignore_index=True).to_csv(ANALYSIS_DIR/'cross_model_accuracy_and_answer_bias.csv',index=False)
    save_json(SHARED_ROOT/'both_models_complete.json',dict(implementation=IMPLEMENTATION_SHA256,
        inputs_sha256=file_hash(SHARED_ROOT/'inputs.json'),primary_family_tests=14,secondary_family_tests=8,
        note='The two multiplicity families are separate; there is no claimed single global 22-test error guarantee.'))
    display(p.round(4));display(s.round(4))
    print('Both models completed. Joint tables are included in this model’s export; neither model is selected by its outcome.')


def now_utc():
    return datetime.now(timezone.utc).isoformat()

def tensor_identity(inputs):
    result = {}
    for key, value in inputs.items():
        if torch.is_tensor(value):
            x = value.detach().cpu().contiguous()
            result[key] = dict(shape=list(x.shape), dtype=str(x.dtype),
                               sha256=hashlib.sha256(x.view(torch.uint8).numpy().tobytes()).hexdigest())
        else:
            result[key] = str(value)
    return result

def init_runtime():
    global RUNTIME_IDENTITY, RUNTIME_SHA
    import PIL
    import transformers
    assert adapter.metadata['resolved_revision'] == SPEC['revision']
    assert adapter.metadata['dtype'] == 'bfloat16'
    assert not getattr(adapter.model, 'is_quantized', False), 'Use native BF16, not a quantized checkpoint.'
    assert not any(str(v) in ('cpu', 'disk') for v in getattr(adapter.model, 'hf_device_map', {}).values()), 'The full BF16 model must fit on GPU.'
    adapter.model.eval()
    torch.manual_seed(SEED)
    np.random.seed(SEED % (2**32))
    torch.backends.cuda.matmul.allow_tf32 = False
    plans = {}
    for condition in CONDITIONS:
        ex = make_example(next(s for s in SPECS if s['condition'] == condition))
        plan = adapter._candidate_plan(ex)
        assert plan is not None, 'Expected exact single-token candidates after a common prefix.'
        prefix, tokens = plan
        assert set(tokens) == set(LABELS) and len(set(tokens.values())) == 3
        plans[condition] = dict(prefix=list(prefix), token_by_label=tokens)
    assert len({digest(v) for v in plans.values()}) == 1, 'Candidate scoring differs between conditions.'
    template = adapter.processor.chat_template
    tokenizer_template = adapter.processor.tokenizer.chat_template
    processor_config = adapter.processor.to_dict() if hasattr(adapter.processor, 'to_dict') else {}
    processor_config = json.loads(json.dumps(processor_config, default=str))
    image_processor = getattr(adapter.processor, 'image_processor', None)
    image_config = image_processor.to_dict() if image_processor is not None else None
    image_config = json.loads(json.dumps(image_config, default=str))
    RUNTIME_IDENTITY = dict(model_key=MODEL_KEY, model_metadata=adapter.metadata,
        gpu=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        torch=torch.__version__, transformers=transformers.__version__, pillow=PIL.__version__,
        cuda=torch.version.cuda, numpy=np.__version__, pandas=pd.__version__,
        implementation=IMPLEMENTATION_SHA256, dtype='bfloat16', batch_size=1,
        thinking=False, max_new_tokens_generation=16, candidate_plans=plans,
        toolkit={str(p.relative_to(TOOLKIT_ROOT)):file_hash(p) for p in sorted((TOOLKIT_ROOT/'src').rglob('*.py'))},
        protocol_sha256=file_hash(RUN_ROOT/'protocol.json'), phase_protocol_sha256=file_hash(SHARED_ROOT/'phase_protocol.json'),
        support_sha256=file_hash(SHARED_ROOT/'support.json'), inputs_sha256=file_hash(SHARED_ROOT/'inputs.json'),
        processor_config=processor_config, image_processor_config=image_config,
        chat_template=template, tokenizer_chat_template=tokenizer_template)
    RUNTIME_SHA = freeze(PHASE_ROOT/'runtime_identity.json', RUNTIME_IDENTITY)
    print('Frozen candidate plans:', plans['image'])
    print('Direct-answer scoring; hidden reasoning disabled. Results do not measure a reasoning-enabled ceiling.')

def score_one(spec):
    ex = make_example(spec)
    prefix, tokens = adapter._candidate_plan(ex)
    raw_inputs = adapter._template([ex.messages()], add_generation_prompt=True)
    inputs = adapter._append_text_prefix(adapter._to_device(raw_inputs), prefix)
    identity = tensor_identity(inputs)
    present_pixels = any('pixel' in k and torch.is_tensor(v) and v.numel() > 0 for k, v in inputs.items())
    assert present_pixels == (ex.image_path is not None), 'Image/text-only processor input mismatch.'
    started = time.monotonic()
    with torch.inference_mode():
        outputs = adapter.model(**inputs, use_cache=False)
        logits = outputs.logits[0, -1].float()
        logp = torch.log_softmax(logits, dim=-1)
        values = [float(logp[tokens[label]].item()) for label in LABELS]
    assert np.isfinite(values).all()
    predicted = LABELS[int(np.argmax(values))]
    semantic = ex.label_to_semantic[predicted]
    correct = int(predicted == ex.gold_label)
    gold_index = LABELS.index(ex.gold_label)
    margin = values[gold_index] - max(v for i, v in enumerate(values) if i != gold_index)
    sorted_scores = sorted(values, reverse=True)
    row = dict(spec, predicted_label=predicted, predicted_semantic=semantic, correct=correct,
        candidate_logprobs=dict(zip(LABELS, values)), gold_margin=margin,
        predicted_balance=int(semantic == 'balanced'), wrong_follows_weight=int(not correct and semantic == spec['weight_outcome']),
        top_score_tie=sorted_scores[0] == sorted_scores[1],
        prompt_sha256=hashlib.sha256(ex.prompt.encode()).hexdigest(),
        input_sha256=digest(identity), prepared_inputs=identity,
        input_token_ids=inputs['input_ids'][0].detach().cpu().tolist(),
        input_token_count=int(inputs['attention_mask'].sum()),
        scoring_mode='single_token_next_token', candidate_prefix=list(prefix), candidate_tokens=tokens,
        scoring_seconds=time.monotonic()-started, generation_audited=spec['trial_id'] in AUDIT_IDS,
        generated_label=None, raw_response=None, parsed_response=None, generated_correct=None,
        recorded_at_utc=now_utc())
    del outputs, logits, logp, inputs, raw_inputs
    if row['generation_audited']:
        generated = adapter.generate_batch([ex])[0]
        label = strict_label(generated.parsed_content)
        row.update(generated_label=label, raw_response=generated.raw_response,
                   parsed_response=generated.parsed_content,
                   generated_correct=int(label == ex.gold_label) if label is not None else None)
    return row

def run_scoring():
    folder = PHASE_ROOT/'families'
    folder.mkdir(parents=True, exist_ok=True)
    grouped = {f['family_id']: [s for s in SPECS if s['family_id'] == f['family_id']] for f in FAMILIES}
    all_rows = []
    bar = tqdm(total=len(SPECS), desc=f'{MODEL_KEY} / {PHASE}: paired inputs', unit='forward')
    try:
        for family in FAMILIES:
            fid = family['family_id']
            path = folder/(fid+'.json')
            expected = {s['trial_id']:s for s in grouped[fid]}
            rows = []
            identity = digest([RUNTIME_SHA, fid])
            if path.exists():
                old = read_json(path)
                assert old['identity'] == identity, 'Different runtime or input definition; use a new RUN_ID.'
                rows = old['rows']
                assert old['rows_sha256'] == digest(rows), 'Cached result rows changed: ' + str(path)
                assert len({r['trial_id'] for r in rows}) == len(rows)
                for row in rows:
                    assert row['trial_id'] in expected
                    assert all(row[k] == v for k, v in expected[row['trial_id']].items())
                bar.update(len(rows))
            done = {r['trial_id'] for r in rows}
            # Fixed pseudo-random ordering prevents condition/order from being confounded.
            order = sorted(expected, key=lambda key: digest(['trial_order', SEED, key]))
            try:
                for tid in order:
                    if tid in done:
                        continue
                    row = score_one(expected[tid])
                    rows.append(row)
                    bar.update(1)
                    bar.set_postfix_str(expected[tid]['condition'])
                    if len(rows) % 6 == 0:
                        save_json(path, dict(identity=identity, family_id=fid, rows=rows, rows_sha256=digest(rows)))
            finally:
                # Save completed inputs even when an error or interruption occurs.
                save_json(path, dict(identity=identity, family_id=fid, rows=rows, rows_sha256=digest(rows)))
            assert len(rows) == len(expected)
            all_rows.extend(rows)
    finally:
        bar.close()
    return all_rows

def export_support():
    """Works without model weights after the setup/definition cells have run."""
    completion_path = PHASE_ROOT/'completion.json'
    complete = completion_path.exists() and read_json(completion_path).get('status') == 'complete'
    sources = {}
    for name in ('protocol.json', 'historical_exclusion.json'):
        p = RUN_ROOT/name
        if p.exists():
            sources[name] = p
    for p in SHARED_ROOT.rglob('*'):
        if p.is_file() and not p.name.endswith('.tmp'):
            sources['shared/'+PHASE+'/'+p.relative_to(SHARED_ROOT).as_posix()] = p
    for p in PHASE_ROOT.rglob('*'):
        if p.is_file() and not p.name.endswith('.tmp'):
            sources['models/'+MODEL_KEY+'/'+PHASE+'/'+p.relative_to(PHASE_ROOT).as_posix()] = p
    manifest = dict(schema=SCHEMA, model=MODEL_KEY, phase=PHASE,
        status='complete' if complete else 'partial_not_complete',
        created_at_utc=now_utc(), files={name:file_hash(p) for name,p in sources.items()})
    output = RUN_ROOT/f'VisualTorque_Frozen_Visual_Validation_{MODEL_KEY}_Output.zip'
    tmp = output.with_suffix('.zip.tmp')
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, path in sources.items():
            archive.write(path, name)
        archive.writestr('export_manifest.json', json.dumps(manifest, ensure_ascii=False, indent=2))
    with zipfile.ZipFile(tmp) as archive:
        assert archive.testzip() is None
    tmp.replace(output)
    local = Path('/content')/output.name if Path('/content').is_dir() else output
    if local.resolve() != output.resolve():
        shutil.copyfile(output, local)
    print('Export status:', manifest['status'], '\nDrive:', output, '\nDownload:', local)
    try:
        from google.colab import files
        files.download(str(local))
    except ImportError:
        pass
    return str(output)


In [ ]:
%%cell_progress 7/13 Freeze exclusions, scenes and paired prompts
cell_stage('Verifying four known source ZIPs and freezing geometry exclusions')
SOURCE_SNAPSHOT,EXCLUDED=freeze_sources(RUN_ROOT,SOURCE_PATHS,EXPECTED_SOURCE_VERSIONS,EXCLUSION_PATHS)
cell_stage('Generating or reusing the same 96 held-out physical families')
FAMILIES,INPUTS=prepare_validation(SHARED_ROOT,SOURCE_SNAPSHOT,EXCLUDED)
INPUT_BY_ID={x['trial_id']:x for x in INPUTS}
SCENES={s['scene_id']:s for f in FAMILIES for s in f['scenes']}
SPECS=[{k:v for k,v in x.items() if k not in ('prompt','image_filename','image_sha256','generation_audit')} for x in INPUTS]
AUDIT_IDS={x['trial_id'] for x in INPUTS if x['generation_audit']}
show_validation_inputs()


In [ ]:
%%cell_progress 8/13 Check GPU and model access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Use a GPU that can hold the full BF16 model, as in your completed runs.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login unavailable:', type(exc).__name__)


In [ ]:
%%cell_progress 9/13 Load model and check preprocessing identity
cell_stage(f'Loading native BF16 {MODEL_KEY}')
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()
adapter=HuggingFaceMultimodalAdapter(SPEC['model_id'],revision=SPEC['revision'],dtype='bfloat16',device_map='auto')
display(adapter.metadata)
init_validation_runtime()


In [ ]:
%%cell_progress 10/13 Run 8640 frozen paired inputs
ROWS=run_scoring()


In [ ]:
%%cell_progress 11/13 Analyze complete validation
SUMMARY=finish_validation(ROWS)


In [ ]:
%%cell_progress 12/13 Combine both models and apply planned corrections
combine_validation_models()


In [ ]:
%%cell_progress 13/13 Export complete or partial validation output
OUTPUT_ZIP=export_support()
